# 13 Day- End-to-End Architecture Design

In [0]:
%sql
CREATE DATABASE ai_challenge;
USE ai_challenge;

In [0]:
transactions_df = spark.table("workspace.default.data")

transactions_df.display()

transactionID,customerID,franchiseID,dateTime,product,quantity,unitPrice,totalPrice,paymentMethod,cardNumber
1002961,2000253,3000047,2024-05-14T12:17:01.495Z,Golden Gate Ginger,8,3,24,amex,3.78154E14
1003007,2000226,3000047,2024-05-10T23:10:10.239Z,Austin Almond Biscotti,36,3,108,mastercard,2.24463E15
1003017,2000108,3000047,2024-05-16T16:34:10.613Z,Austin Almond Biscotti,40,3,120,mastercard,2.49057E15
1003068,2000173,3000047,2024-05-02T04:31:51.612Z,Pearly Pies,28,3,84,amex,3.43809E14
1003103,2000075,3000047,2024-05-04T23:44:26.902Z,Pearly Pies,28,3,84,visa,4.37708E15
1003147,2000295,3000047,2024-05-15T16:17:06.259Z,Austin Almond Biscotti,32,3,96,amex,3.71094E14
1003196,2000237,3000047,2024-05-07T11:13:22.469Z,Tokyo Tidbits,40,3,120,mastercard,5.53881E15
1003329,2000272,3000047,2024-05-06T03:32:16.017Z,Outback Oatmeal,28,3,84,visa,4.87248E15
1001264,2000209,3000047,2024-05-16T17:32:28.547Z,Pearly Pies,28,3,84,mastercard,5.28711E15
1001287,2000120,3000047,2024-05-15T08:41:28.406Z,Austin Almond Biscotti,40,3,120,amex,3.76211E14


Data Ingestion (Bronze Layer)

In [0]:
transactions_df = spark.table("workspace.default.data")

In [0]:
Create Silver Layer

In [0]:
silver_df = transactions_df.dropna().dropDuplicates()

silver_df.write.format("delta") \
        .mode("overwrite") \
        .saveAsTable("workspace.default.silver_transactions")

Silver Layer (Clean Data)

In [0]:
silver_df = transactions_df.dropna().dropDuplicates()

silver_df.write.format("delta") \
    .mode("overwrite") \
    .saveAsTable("workspace.default.silver_transactions")

Gold Layer (Feature Engineering)

1️. Extract Time-Based Features

In [0]:
from pyspark.sql.functions import hour, dayofweek, month

gold_df = silver_df \
    .withColumn("hour", hour("dateTime")) \
    .withColumn("day_of_week", dayofweek("dateTime")) \
    .withColumn("month", month("dateTime"))

2. Create High Value Transaction Flag

In [0]:
from pyspark.sql.functions import col

gold_df = gold_df.withColumn(
    "high_value",
    (col("totalPrice") > 5000).cast("int")
)

3. Encode Categorical Columns

In [0]:
from pyspark.ml.feature import StringIndexer

product_indexer = StringIndexer(inputCol="product", outputCol="product_index")
payment_indexer = StringIndexer(inputCol="paymentMethod", outputCol="payment_index")

gold_df = product_indexer.fit(gold_df).transform(gold_df)
gold_df = payment_indexer.fit(gold_df).transform(gold_df)

5. Create Feature Vector

In [0]:
from pyspark.ml.feature import VectorAssembler

feature_cols = [
    "quantity",
    "unitPrice",
    "hour",
    "day_of_week",
    "month",
    "product_index",
    "payment_index"
]

assembler = VectorAssembler(
    inputCols=feature_cols,
    outputCol="features"
)

gold_df = assembler.transform(gold_df)

6.Save Gold Layer

In [0]:
gold_df.write.format("delta") \
    .mode("overwrite") \
    .saveAsTable("workspace.default.gold_transactions")

# 13 Day- End-to-End Architecture Design

##  All layers Show :

In [0]:
from pyspark.sql.functions import lit


bronze_df = spark.table("workspace.default.data")
silver_df = spark.table("workspace.default.silver_transactions")
gold_df = spark.table("workspace.default.gold_transactions")


bronze_count = bronze_df.count()
silver_count = silver_df.count()
gold_count = gold_df.count()

summary_df = spark.createDataFrame([
    ("Bronze Layer", "workspace.default.data", bronze_count),
    ("Silver Layer", "workspace.default.silver_transactions", silver_count),
    ("Gold Layer", "workspace.default.gold_transactions", gold_count)
], ["Layer", "Table Name", "Record Count"])


summary_df.display()

Layer,Table Name,Record Count
Bronze Layer,workspace.default.data,3333
Silver Layer,workspace.default.silver_transactions,3333
Gold Layer,workspace.default.gold_transactions,3333


##  task 2. Pipeline Flow Documentation

### End-to-End Pipeline Flow

### Step 1: Data Ingestion

Raw transactional data is loaded from Unity Catalog table workspace.default.data. This acts as the Bronze layer.

### Step 2: Data Cleaning

Data is cleaned by removing null values and duplicates. Stored as silver_transactions.

### Step 3: Feature Engineering

Time-based features (hour, day_of_week, month) are extracted.
Categorical columns are encoded.
Binary target column high_value is created.

Stored as gold_transactions.

### Step 4: Model Training

Gold dataset is used for training classification model.

### Step 5: Model Tracking

MLflow is used to log parameters, metrics, and model artifacts.

### Step 6: Deployment

Best model is registered in Model Registry.

### Step 7: Monitoring & Retraining

Model performance is monitored. Retraining is triggered if performance degrades.

## task 3. Retraining Strategy

1. Retrain model if accuracy drops below 80%.

2. Schedule monthly retraining using Databricks Workflows.

3. Compare new model performance with production model.

4. Promote new model only if performance improves.

5. Maintain version history in MLflow Model Registry.